<a href="https://colab.research.google.com/github/tanishataranoon/Game/blob/main/Toxic_Text_Classification_Comparison_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
import re

# Load dataset
from https://huggingface.co/datasets/thesofakillers/jigsaw-toxic-comment-classification-challenge?library=datasets


In [ ]:
from datasets import load_dataset

df = load_dataset("thesofakillers/jigsaw-toxic-comment-classification-challenge")

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 159571
    })
    test: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 306328
    })
})

In [ ]:
train_df = df['train'].to_pandas()
test_df = df['test'].to_pandas()

Train data size and trained data

In [ ]:
print(train_df.shape)
print(train_df.head())

(159571, 8)
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
2  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   
3  0001b41b1c6bb37e  "\nMore\nI can't make any real suggestions on ...      0   
4  0001d958c54c6e35  You, sir, are my hero. Any chance you remember...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
2             0        0       0       0              0  
3             0        0       0       0              0  
4             0        0       0       0              0  


Test data size and test data

In [ ]:
print(test_df.shape)
print(test_df.head())

(306328, 8)
                 id                                       comment_text  toxic  \
0  00001cee341fdb12  Yo bitch Ja Rule is more succesful then you'll...    NaN   
1  0000247867823ef7  == From RfC == \n\n The title is fine as it is...    NaN   
2  00013b17ad220c46  " \n\n == Sources == \n\n * Zawe Ashton on Lap...    NaN   
3  00017563c3f7919a  :If you have a look back at the source, the in...    NaN   
4  00017695ad8997eb          I don't anonymously edit articles at all.    NaN   

   severe_toxic  obscene  threat  insult  identity_hate  
0           NaN      NaN     NaN     NaN            NaN  
1           NaN      NaN     NaN     NaN            NaN  
2           NaN      NaN     NaN     NaN            NaN  
3           NaN      NaN     NaN     NaN            NaN  
4           NaN      NaN     NaN     NaN            NaN  


**Raw data from the trained dataset before any pre processing **

In [ ]:
train_df[['comment_text', "toxic"]].head(6)

,comment_text,toxic
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0
5,"""\n\nCongratulations from me as well, use the ...",0


# PreProcessing

**Basic Preprocessing**



1.   Lower case




In [ ]:
def to_lower(text):
    return text.lower()


2.   URL remove



In [ ]:
def remove_urls(text):
    import re
    return re.sub(r"http\S+|www\S+", "", text)


3. Mentions remove



In [ ]:
def remove_mentions(text):
    import re
    return re.sub(r"@\w+", "", text)



4. Remove Punctuation



In [ ]:
def remove_punctuation(text):
    import re
    return re.sub(r"[^a-zA-Z\s]", "", text)


5. Remove extra space




In [ ]:
def remove_extra_spaces(text):
    import re
    return re.sub(r"\s+", " ", text).strip()

6. Stopword removed

In [ ]:
def stopword_removal(text):
    stopwords = {"the", "is", "a", "an"}
    return " ".join([w for w in text.split() if w not in stopwords])

**Advance Preprocessing**

In [ ]:
!pip install emoji

1. Emoji to text

In [ ]:
import emoji

def emoji_to_text(text):
  return emoji.demojize(text, delimiters=(" ", " "))

 2. Convert to full forms



In [ ]:
def expand_contractions(text):
  contractions = {
    "ain't": "is not",
    "aren't": "are not",
    "can't": "cannot",
    "can't've": "cannot have",
    "'cause": "because",
    "don't": "do not",
    "you're": "you are",
    "it's": "it is",
    "i'm": "i am",
    "won't": "will not"
  }
  for k,v in contractions.items():
      text = text.replace(k,v)
      return text



3. Normalize repetitions\
stuuuuupid → stupid

In [ ]:
def normalize_repeated_chars(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

4. Bad Word Normalization\
b!tch = bitch

In [ ]:
def normalize_obscene(text):
    text = text.replace("b!tch", "bitch")
    text = text.replace("f*ck", "fuck")
    text = text.replace("a$$", "ass")
    return text

PreProcess function

In [ ]:
def basic_preprocess(text):
    text = to_lower(text)
    text = remove_urls(text)
    text = remove_mentions(text)
    text = remove_punctuation(text)
    text = remove_extra_spaces(text)
    return text

In [ ]:
def advanced_preprocess(text):
    text = emoji_to_text(text)
    text = expand_contractions(text)
    text = normalize_obscene(text)
    text = normalize_repeated_chars(text)
    text = to_lower(text)
    text = remove_urls(text)
    text = remove_mentions(text)
    text = remove_punctuation(text)
    text = remove_extra_spaces(text)
    return text

# Apply the preprocessing

Before Applying

In [ ]:
pd.set_option('display.max_colwidth', None)
train_df["raw"] = train_df["comment_text"]
train_df[["raw"]].sample(5)


,raw
103475,"Please provide diffs, as I am not going to respond by guessing what two incidents you're talking about. It seems like you're repeating something you heard without checking the facts carefully. Talk"
70912,"I know they are (and they are also both in the British Isles, of course), but again - if you want GB, find a source."
63009,Where have you seen any personal attacks? You just personally prefer to offend me. What can I do about it? I don't know. And yes I don't want to continue any conversations with ignorant mazapukers. Poor those people who discussed that with you and similar to you. I understand them very well. Are you so unintellegent mazapuker? Have you ever thought if there won't be any problem you would never see me here Ukrainian a citizen of Ukraine and a resident of Kyiv telling you that? Like African American that trying to make someone to stop calling him N. Very sad you are not liberal enough to understand that. It is inevitable but your conservative mind wants to stop it. Everything changes. After your death (if Wikipedia will exist) it for sure will be Kyiv and you won't exist by that time and couldn't change it. Why are you trying to stop things which you cannot change? What kind of perversion is it? 94.244.129.207
53773,"""\nWhile I understand the function of WP:ACC, I'm confused as to your question - """"a specific group""""? What do you mean? | 39 """
45820,"""\n\nHey, nobody's wrong here, and nobody is defininitely a loser. I think that user Abhishek191288 reverted Markshen1985's edits because there are some opinions to his edits such as logo is a brilliant red kapok delicately adoring a blue vertical tail fin. Also, much of """"History and Development is unsourced. Having said that, Markshen1985's work is remarkable. Also, there seems to be a touch of discrimination here, with """"Chinese"""" being thrown around. I'm going report this to . (Talk) (Contributions)(Feed back needed @ Talk page) """


Applying Basic Preprocessing

In [ ]:

train_df["basic_clean"] = train_df["raw"].apply(basic_preprocess)
train_df[["basic_clean"]].sample(5)

,basic_clean
44469,dont be barbarian this picture is older on hungarian wikipedia than your registration sometimes some pictures from hungarian wikipedia are unable to insert to english wikipedia i dont know why so i uploaded it
41678,more false accusations without proof it really must have hurt
84440,i did source it then you deleted it so i added it again the source i added was an interview in which gaga herself confirms the new track named heavy metal lover i even had the decency to tell you at which point she says this its not my fault youre ignorant either understand that youre ruining wikipedia by deleting everything that is official and confirmed or let other users add information
78394,september utc ive gone the next step and done so
79274,actually the winner of the dallas vs seattle game doesnt automatically play either new orleans or chicago its based on both wild card game winners to determine who plays its the vs game that is more determined if the seeds wins they play new orleans if the seed wins they play chicago based on that dallasseattle plays the other team


Comparison Of Raw and Basic Clean by Basic Preprocessing

In [ ]:
pd.set_option('display.max_colwidth', None)
train_df[["raw", "basic_clean"]].sample(5)

,raw,basic_clean
141620,"""\nIf it's meant as an experiment, how about putting it in your userspace? Deletion criterion #2 reads """"Templates should not be redundant"""". I don't believe there are many templates that have multiple versions; could you please back that claim with evidence? adiant_>|< 10:00, Jun 19, 2005 (UTC)""",if its meant as an experiment how about putting it in your userspace deletion criterion reads templates should not be redundant i dont believe there are many templates that have multiple versions could you please back that claim with evidence adiant jun utc
80426,"""\n\n Libya casualty report (French operations) \n\nYou claimed that Libya sources are unreliable, and delete my contribution... And western reports are? You're a biased fuck. So, the US, UK, or France can claim whatever casualties they want... thats fine, but when Libya claims to have caused casuatlies you immediately lable it """"unreliable"""" and remove it? Your bias is showing asswipe... cover it up before you stain Wikipedia's name even more.- AGSman61""",libya casualty report french operations you claimed that libya sources are unreliable and delete my contribution and western reports are youre a biased fuck so the us uk or france can claim whatever casualties they want thats fine but when libya claims to have caused casuatlies you immediately lable it unreliable and remove it your bias is showing asswipe cover it up before you stain wikipedias name even more agsman
141980,"Nope, I'm really just saying you've got a clause with a colon at the end that says 4 groups are following and then a list with 5 things in it. If you change the colon to a period it looks kind of stupid. I can count to four and I can count to five and, well they aren't the same thing.",nope im really just saying youve got a clause with a colon at the end that says groups are following and then a list with things in it if you change the colon to a period it looks kind of stupid i can count to four and i can count to five and well they arent the same thing
98812,"""\n\n Bhadani \n\nHi Dab. Sorry to take up an issue that refuses to die. While I understand your frustration, and have tried to explain your case to the people involved (here, for example), I feel that you should've A'edGF a bit instead of presuming that Bhadani was a troll or a newbie. He is neither. He's nationalistic, verbose (like most Indians like me), but a good faith editor. In my opinion, you should say sorry to him just for presuming him to be a """"troll"""". I know it may be difficult given the positions the parties have taken, but it would do good. Thanks. \talk \contribs """,bhadani hi dab sorry to take up an issue that refuses to die while i understand your frustration and have tried to explain your case to the people involved here for example i feel that you shouldve aedgf a bit instead of presuming that bhadani was a troll or a newbie he is neither hes nationalistic verbose like most indians like me but a good faith editor in my opinion you should say sorry to him just for presuming him to be a troll i know it may be difficult given the positions the parties have taken but it would do good thanks talk contribs
130292,"These issues are best decided through discussion. Perhaps one of you can open a talk page discussion so that others can provide their input? Cheers,",these issues are best decided through discussion perhaps one of you can open a talk page discussion so that others can provide their input cheers


Applying Advanced Preprocessing

In [ ]:
train_df["advanced_clean"] = train_df["raw"].apply(advanced_preprocess)
train_df[["advanced_clean"]].sample(5)

,advanced_clean
24355,youre a fucking liar who doesnt know how to do...
41272,with what software did you create the picture
120606,the smolensk conference can you complete the i...
13727,this is the exact quote thus we can begin by l...
21403,theres also way too much pov in this article s...


Comparison of Raw and Advanced Clean by Advanced Preprocessing


In [ ]:
pd.set_option('display.max_colwidth', None)
train_df[["raw","advanced_clean"]].sample(5)

,raw,advanced_clean
106794,"It IS true, and although you may not believe it, I realise now that it is not notable and therby does not belong. Perhaps another more notable event in the future will occur that deserves a place on the Wiki.",it is true and although you may not believe it i realise now that it is not notable and therby does not belong perhaps another more notable event in the future will occur that deserves a place on the wiki
115444,"This anonymous user has been blocked (according to a notice on the talk page). Perhaps the semi-protection expired? Is there a process for reapplying? Given the two recent anonymous vandalisms, it might be appropriate.",this anonymous user has been blocked according to a notice on the talk page perhaps the semiprotection expired is there a process for reapplying given the two recent anonymous vandalisms it might be appropriate
87931,"Vandalism after you gave final warning\nI'm not sure the best place to report this, but I saw on User talk:Yankee412 that you had given a last warning about vandalism. I've just posted another vandalism warning there. (BTW, where should I really be posting this?)",vandalism after you gave final warning im not sure the best place to report this but i saw on user talkyankee that you had given a last warning about vandalism ive just posted another vandalism warning there btw where should i really be posting this
34467,"""\n\n Help me! \n\nPlease help me with...\nI was instantly deleted I just saved the page for a second as I was uploading images. I know I need to add sources, verifiable accounts etc. I have created quite a few pages, notably David Carol's. Please give me a moment to finish. It is a huge task!! Thank you!\nMichele """"shell4art"""" Luccketta\n\n """,help me please help me with i was instantly deleted i just saved the page for a second as i was uploading images i know i need to add sources verifiable accounts etc i have created quite a few pages notably david carols please give me a moment to finish it is a huge task thank you michele shellart luccketta
9745,Your question at the Help desk,your question at the help desk
